# KojoBench SFT — Qwen2.5-7B-Instruct

LoRA fine-tune on KojoBench task pairs.  
- Model: `unsloth/Qwen2.5-7B-Instruct`, 4-bit  
- LoRA: r=8, alpha=16, dropout=0.05, all projection modules  
- Training: 15 epochs, effective batch 8 (per_device=2 × grad_accum=4), lr=2e-4  
- Adapter saved to `/content/drive/MyDrive/kojo-lora`

**Before running:** ensure KojoBench is at `MyDrive/KojoBench/` in your Drive.

In [ ]:
%%capture
!pip install unsloth trl datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/KojoBench')
assert BASE.exists(), f'KojoBench not found at {BASE}. Check your Drive path.'
print(f'KojoBench found at {BASE}')

In [ ]:
import ast

eval_source = (BASE / 'eval' / 'eval_kojobench2.py').read_text(encoding='utf-8')
tree = ast.parse(eval_source)
SYSTEM_PROMPT = None
for node in ast.walk(tree):
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == 'SYSTEM_PROMPT':
                SYSTEM_PROMPT = ast.literal_eval(node.value)
                break

assert SYSTEM_PROMPT, 'SYSTEM_PROMPT not found in eval_kojobench2.py'
print(f'SYSTEM_PROMPT extracted: {len(SYSTEM_PROMPT)} chars')
print(SYSTEM_PROMPT[:120] + '...')

In [ ]:
import re


def extract_raw_commands(kojo_code):
    """Reverse _wrap_in_picture: pull inner drawing commands out of a KojoTask file."""
    lines = kojo_code.strip().splitlines()
    pic_start = pic_end = None
    for i, line in enumerate(lines):
        s = line.strip()
        if s == 'def shape = Picture {':
            pic_start = i
        elif pic_start is not None and i > pic_start and s == '}':
            pic_end = i
            break
    if pic_start is not None and pic_end is not None:
        inner = lines[pic_start + 1 : pic_end]
        return '\n'.join(l[4:] if l.startswith('    ') else l for l in inner).strip()
    # Fallback: strip wrapper-only lines, return the rest
    skip = {'cleari()', 'clear()', 'drawCentered(shape)'}
    keep = [l for l in lines
            if l.strip() not in skip
            and not re.match(r'^[ \t]*setSpeed\s*\(', l)]
    return '\n'.join(keep).strip()


def make_geometry(query, raw_commands):
    cmd_lines = [l.strip() for l in raw_commands.splitlines()
                 if l.strip() and not l.strip().startswith('//')]
    first_cmd = cmd_lines[0][:80] if cmd_lines else 'turtle commands'
    parts = [
        '<geometry>',
        f'Goal: {query.strip()}',
        'Canvas/scale: Approximately 500x500px, turtle at (0,0) heading North.',
        'Main shapes:',
        f'- Shape built via turtle arc/movement commands. First: {first_cmd}',
        'Drawing order:',
        '1. Configure pen (color, thickness) if needed.',
        '2. Set initial heading if needed.',
        '3. Execute movement and arc commands to draw the shape.',
        '</geometry>',
    ]
    return '\n'.join(parts)


def make_assistant_turn(query, raw_commands):
    geo = make_geometry(query, raw_commands)
    return f'{geo}\n\n```scala\n{raw_commands}\n```'


conversations = []
skipped = []
benchmark_dir = BASE / 'benchmark'

task_dirs = sorted(
    [d for d in benchmark_dir.iterdir()
     if d.is_dir() and re.match(r'^Task\d+$', d.name)],
    key=lambda p: int(re.search(r'\d+', p.name).group())
)

for task_dir in task_dirs:
    n = re.search(r'\d+', task_dir.name).group()
    query_path = task_dir / f'KojoQuery{n}.md'
    kojo_path  = task_dir / f'KojoTask{n}.kojo'
    if not query_path.exists() or not kojo_path.exists():
        skipped.append((task_dir.name, 'missing file'))
        continue
    query        = query_path.read_text(encoding='utf-8').strip()
    raw_kojo     = kojo_path.read_text(encoding='utf-8')
    raw_commands = extract_raw_commands(raw_kojo)
    if not raw_commands:
        skipped.append((task_dir.name, 'empty after extraction'))
        continue
    conversations.append({
        'conversations': [
            {'from': 'system', 'value': SYSTEM_PROMPT},
            {'from': 'human',  'value': query},
            {'from': 'gpt',    'value': make_assistant_turn(query, raw_commands)},
        ]
    })

print(f'Valid tasks: {len(conversations)}')
if skipped:
    print(f'Skipped ({len(skipped)}): {skipped}')

In [ ]:
sample = conversations[0]['conversations']
print('=== SYSTEM (first 150 chars) ===')
print(sample[0]['value'][:150] + '...')
print('\n=== HUMAN ===')
print(sample[1]['value'])
print('\n=== GPT (assistant turn) ===')
print(sample[2]['value'])

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096  # system prompt ~1800 tokens; 4096 gives safe headroom

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # auto: bfloat16 on A100/L4, float16 on T4
    load_in_4bit=True,
)
print('Model loaded.')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

_ROLE_MAP = {'system': 'system', 'human': 'user', 'gpt': 'assistant'}

def _to_messages(convs):
    return [{'role': _ROLE_MAP[c['from']], 'content': c['value']} for c in convs]

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(
            _to_messages(convs),
            tokenize=False,
            add_generation_prompt=False,
        )
        for convs in examples['conversations']
    ]
    return {'text': texts}

hf_dataset = Dataset.from_list(conversations)
hf_dataset = hf_dataset.map(apply_template, batched=True)

n           = len(hf_dataset)
eff_batch   = 2 * 4
steps_epoch = -(-n // eff_batch)  # ceiling division
print(f'Dataset       : {n} examples')
print(f'Effective batch: {eff_batch}')
print(f'Steps / epoch  : {steps_epoch}')
print(f'Total steps    : {steps_epoch * 15}  (15 epochs)')
print(f'\nFormatted sample (first 400 chars):')
print(hf_dataset[0]['text'][:400])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=15,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='/content/kojo-checkpoints',
        save_strategy='no',
        report_to='none',
    ),
)

trainer_stats = trainer.train()
print(f'\nDone. Runtime: {trainer_stats.metrics["train_runtime"]:.0f}s  '
      f'Loss: {trainer_stats.metrics.get("train_loss", "N/A")}')

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/kojo-lora'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Adapter saved to {SAVE_PATH}')
print(f'Files: {sorted(str(p.name) for p in Path(SAVE_PATH).iterdir())}')